In [1]:
import pandas as pd
import os
import numpy as np
from scipy.cluster.vq import kmeans, whiten
from scipy.spatial.distance import pdist,squareform
from scipy.cluster.hierarchy import linkage, fcluster

In [2]:
anime_to_consider = pd.read_csv("AnimeWithImages.csv")
embeddings_dir="Embeddings"
os.makedirs(embeddings_dir,exist_ok=True)

In [3]:
def cluster(results_dir,obs):
    obs = whiten(obs)
    os.makedirs(results_dir, exist_ok=True)
    cluster_amounts = range(5, 31)
    RNG = 1564
    methods = "complete", "average", 'weighted'
    for method in methods:
        linkage_matrix = linkage(obs.copy(), method=method,metric="cosine")
        for c_amount in cluster_amounts:
            c = fcluster(linkage_matrix, c_amount, "maxclust")
            cluster_df = pd.DataFrame({"cluster": c}, index=anime_to_consider["anime_id"])
            cluster_df.to_csv(f"{results_dir}/{method}-cosine_{c_amount}_clusters.csv")
    for c_amount in cluster_amounts:
        centroids,_ = kmeans(obs, c_amount,rng=RNG)
        diff = obs[:,np.newaxis,:]-centroids
        c=np.argmin(np.linalg.norm(diff,axis=2), axis=1)+1
        cluster_df = pd.DataFrame({"cluster": c}, index=anime_to_consider["anime_id"])
        cluster_df.to_csv(f"{results_dir}/K-means_{c_amount}_clusters.csv")
    # AMOUNT = 15
    # epsilons = np.linspace(1,0,AMOUNT,endpoint=False)
    # for epsilon in epsilons:
    #     c = DBSCAN(eps=epsilon,metric="cosine")
    #     c.fit(obs)
    #     cluster_df = pd.DataFrame({"cluster": c.labels_}, index=anime_to_consider["anime_id"])
    #     cluster_df.to_csv(f"{results_dir}/Clustering_DBSCAN-epsilon={epsilon:.2f}-cosine.csv")
    # AMOUNT = 40
    # MAXIMUM = 20
    # epsilons = np.linspace(MAXIMUM,0,AMOUNT,endpoint=False)
    # for epsilon in epsilons:
    #     c = DBSCAN(eps=epsilon)
    #     c.fit(obs)
    #     cluster_df = pd.DataFrame({"cluster": c.labels_+1}, index=anime_to_consider["anime_id"])
    #     cluster_df.to_csv(f"{results_dir}/Clustering_DBSCAN-epsilon={epsilon:.2f}-euclidean.csv")



In [4]:
feature_amount = 100
u,s= (np.load("SVDResults/U_svd_matrix.npy"),
      np.load("SVDResults/S_svd_matrix.npy"))
us_k=u[:,:feature_amount].dot(np.diag(s[:feature_amount]))
results_dir = "SVDClusteringResults/"
print(np.sum(s[:feature_amount])/s.sum())
cluster(results_dir,us_k)

np.save(f"{embeddings_dir}/SVDEmbeddings.npy",us_k)
pairwise_distances= squareform(pdist(us_k,metric="cosine"))
np.save(f"{embeddings_dir}/SVDPairwiseDistances.npy",
        pairwise_distances)

0.22594548986214413


In [5]:
results_dir = "VGGClusteringResults"
image_embeddings = np.load(f"{results_dir}/embeddings.npy")
image_embeddings -= image_embeddings.mean(axis=0)
u,s,v = np.linalg.svd(image_embeddings)
us_k = u[:, :feature_amount] @ np.diag(s[:feature_amount])
print(np.sum(s[:feature_amount])/s.sum())
cluster(results_dir,us_k)
np.save(f"{embeddings_dir}/VGGEmbeddings.npy",us_k)
pairwise_distances= squareform(pdist(us_k,metric="cosine"))
np.save(f"{embeddings_dir}/VGGPairwiseDistances.npy",
        pairwise_distances)

0.40103613463925597


In [6]:
results_dir = "EFFNetClusteringResults"
image_embeddings = np.load(f"{results_dir}/embeddings.npy")
image_embeddings -= image_embeddings.mean(axis=0)
u,s,v = np.linalg.svd(image_embeddings)
us_k = u[:, :feature_amount] @ np.diag(s[:feature_amount])
print(np.sum(s[:feature_amount])/s.sum())
cluster(results_dir,us_k)

np.save(f"{embeddings_dir}/EFFNetEmbeddings.npy",us_k)
pairwise_distances= squareform(pdist(us_k,metric="cosine"))
np.save(f"{embeddings_dir}/EFFNetPairwiseDistances.npy",
        pairwise_distances)

0.2762791921485682


In [13]:
results_dir = "ResNetClusteringResults"
image_embeddings = np.load(f"{results_dir}/embeddings.npy")
image_embeddings -= image_embeddings.mean(axis=0)
u,s,v = np.linalg.svd(image_embeddings)
us_k = u[:, :feature_amount] @ np.diag(s[:feature_amount])
print(np.sum(s[:feature_amount])/s.sum())
cluster(results_dir,us_k)

np.save(f"{embeddings_dir}/ResNetEmbeddings.npy",us_k)
data_frame = pd.DataFrame(us_k,index = anime_to_consider["anime_id"])
data_frame.to_csv(f"{embeddings_dir}/ResNetEmbeddings.csv")
pairwise_distances= squareform(pdist(us_k,metric="cosine"))
data_frame = pd.DataFrame(pairwise_distances,
                          index = anime_to_consider["anime_id"],
                          columns = anime_to_consider["anime_id"])
np.save(f"{embeddings_dir}/ResNetPairwiseDistances.npy",
        pairwise_distances)
data_frame.to_csv(f"{embeddings_dir}/ResNetPairwiseDistances.csv")

0.2506626895981952
